# DSBM Training — Sentinel-1 → Sentinel-2

Run this notebook on **Kaggle** with the `sentinel12-image-pairs-segregated-by-terrain` dataset attached.

**Setup steps before running:**
1. Go to https://www.kaggle.com/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain
2. Click **New Notebook**
3. In the notebook: go to **Settings** (right panel) → **Accelerator** → select **GPU T4 x2**
4. Paste this notebook's contents in and run each cell


In [ ]:
# Step 1: Check GPU
import torch
if torch.cuda.is_available():
    print('GPU ready:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — enable it in Settings > Accelerator')

In [ ]:
# Step 2: Clone the code
!git clone https://github.com/smilingsmilee/dsbm-pytorch.git
import os
os.chdir('dsbm-pytorch')
print('Files:', os.listdir('.'))

In [ ]:
# Step 3: Install dependencies
!pip install -q hydra-core accelerate POT matplotlib seaborn torchmetrics torchdiffeq pytorch-lightning h5py torch-fidelity

In [ ]:
# Step 4: Set up data folders from the Kaggle dataset
import glob, shutil, os

os.makedirs('data/custom_source/images', exist_ok=True)
os.makedirs('data/custom_target/images', exist_ok=True)

# Find S1 (source) and S2 (target) images in the Kaggle dataset
dataset_root = '/kaggle/input/sentinel12-image-pairs-segregated-by-terrain'

s1_files = sorted(glob.glob(f'{dataset_root}/**/s1/**/*.png', recursive=True) +
                  glob.glob(f'{dataset_root}/**/*s1*.png', recursive=True))
s2_files = sorted(glob.glob(f'{dataset_root}/**/s2/**/*.png', recursive=True) +
                  glob.glob(f'{dataset_root}/**/*s2*.png', recursive=True))

# Deduplicate
s1_files = sorted(set(s1_files))
s2_files = sorted(set(s2_files))

print(f'Found {len(s1_files)} S1 images and {len(s2_files)} S2 images')
if s1_files: print('S1 example:', s1_files[0])
if s2_files: print('S2 example:', s2_files[0])

In [ ]:
# Step 5: Copy images into project data folders (uses symlinks to avoid duplicating 700MB)
for f in s1_files:
    dst = f'data/custom_source/images/{os.path.basename(f)}'
    if not os.path.exists(dst):
        os.symlink(f, dst)

for f in s2_files:
    dst = f'data/custom_target/images/{os.path.basename(f)}'
    if not os.path.exists(dst):
        os.symlink(f, dst)

print(f'Source images: {len(os.listdir("data/custom_source/images"))}')
print(f'Target images: {len(os.listdir("data/custom_target/images"))}')

In [ ]:
# Step 6: Run training
!python main.py dataset=custom_transfer method=dbdsb gamma_min=0.034 gamma_max=0.034 LOGGER=CSV device=cuda

In [ ]:
# Step 7: Save results to Kaggle output (so you can download them)
import shutil
shutil.copytree('experiments', '/kaggle/working/experiments', dirs_exist_ok=True)
print('Results saved to Kaggle output — use the Output tab to download')